# Feature extraction

This notebook is my exploration log for feature engineering. I build features for each attempt (one row per user, genuine/impostor label and test) and check their usefulness on simple per-user baseline classifiers, mainly logistic regression, before further model development.

After the EDA I wrote three functions in the `behavioral_ml` package (`src/behavioral_ml`):
- `load_data` (`etl` module): loads all 88 JSON files into two event-level DataFrames, `mouse_df` and `key_df`, indexed by `User ID`, `Is Genuine`, `Test ID` and `Event ID` (`mouse_df` also by `Movement ID`). It drops the columns not needed here and the empty trailing records found in the EDA;
- `add_mouse_kinematics` (`features.mouse` module): calculates kinematics within each mouse movement: step distance, velocity, acceleration and jerk (the third derivative of position);
- `derive_per_movement_features` (`features.mouse` module): aggregates the kinematics to one row per movement.

Sections:
1. Mouse movement features
2. Mouse click and scroll features
3. Keyboard features (first pass)
4. Baseline models
5. Cleaning the keyboard events: one row per physical key press
6. Keyboard features on the cleaned events, and re-evaluation

The cells are kept in the order I worked. The cleaned-up versions of these steps are being moved into `src/behavioral_ml/features/mouse|keyboard.py`, and model evaluation continues in `models.ipynb`.

Let's load the data first using the `load_data` function.
**Exploration record:** the naive stages are retained to explain the design.
Their observations are historical, not the current results table. The corrected
package implementations and model notebook are authoritative for comparisons.


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from behavioral_ml.etl import load_data
from behavioral_ml.features import key, build_feature_sets

raw_mouse_df, raw_key_df = load_data(Path(os.environ.get("KMT_DATA_DIR", "../data/raw_kmt_dataset")))
mouse_df, key_df = raw_mouse_df.copy(), raw_key_df.copy()
mouse_df.head()


Event  ...              Epoch
User ID Is Genuine Test ID Event ID Movement ID            ...                   
1       True       1       0        1            movement  ...  1644173827.568144
                           1        1            movement  ...  1644173827.580155
                           2        1            movement  ...  1644173827.592166
                           3        1            movement  ...  1644173827.605178
                           4        1            movement  ...   1644173827.61819

[5 rows x 3 columns]

## 1. Mouse movement features

`add_mouse_kinematics` splits `Coordinates` into `X` and `Y` and, within each movement (`Movement ID`), takes differences between consecutive `movement` events:
- `dX`, `dY`: change in position (pixels);
- `dS`: distance between consecutive points;
- `dT`: time between consecutive events (seconds);
- `V = dS / dT`: velocity;
- `A = ΔV / dT`: acceleration;
- `Jerk = ΔA / dT`: jerk.

Other event types (clicks, scrolls) get `NaN` in these columns.

In [2]:
from behavioral_ml.features.mouse import add_mouse_kinematics, derive_per_movement_features

movement_features = mouse_df.pipe(add_mouse_kinematics)
movement_features.head()

Event  ...             Jerk
User ID Is Genuine Test ID Event ID Movement ID            ...                 
1       True       1       0        1            movement  ...             <NA>
                           1        1            movement  ...             <NA>
                           2        1            movement  ...             <NA>
                           3        1            movement  ... -45737335.863801
                           4        1            movement  ...   30008559.98049

[5 rows x 12 columns]

In [3]:
movement_features = movement_features.pipe(derive_per_movement_features)
movement_features.head()

Path  ...       V_avg
User ID Is Genuine Test ID Movement ID               ...            
1       False      1       1            1343.113141  ...  306.668178
                           2             738.613262  ...  309.211673
                           4              12.650282  ...    15.65063
                           7             288.162445  ...  168.511605
                           9             127.660290  ...  175.285458

[5 rows x 12 columns]

`derive_per_movement_features` aggregates the kinematics to one row per movement. Besides quantiles of velocity (`V_q10`, `V_median`, `V_q90`) and of acceleration and jerk (the 20th percentile of negative values, `A_min_q20` and `Jerk_min_q20`, and the 80th percentile of positive values, `A_plus_q80` and `Jerk_plus_q80`), I have added a few features:
- `Path`: total distance travelled during the movement;
- `Duration`: movement duration;
- `Displacement`: straight-line (L2) distance between the start and end points;
- `Straightness`: `Displacement / Path`, between 0 and 1; the closer to 1, the straighter the movement;
- `V_avg`: average velocity, `Path / Duration`. I prefer it to averaging instantaneous velocities, because event logging isn't perfect: some events are missing and the base interval between events is about 12 ms, so instantaneous values are noisy.

Very short movements (`Path` below 5 pixels, the `min_Path` parameter) are discarded.

Next, I aggregate the per-movement features to one row per attempt. An attempt has only about 7 movements (see the EDA), so for path length and duration I use the median and the IQR, which are less sensitive to a single unusually long movement. At this level, `V_avg` becomes `V_real_mean` and `V_real_std`.

In [4]:
movement_test_level_features = movement_features.groupby(level=["User ID", "Is Genuine", "Test ID"]).agg(
    Path_median = ("Path", "median"),
    Path_iqr = ("Path", lambda x: x.quantile(.75) - x.quantile(.25)),
    Duration_median = ("Duration", "median"),
    Duration_iqr = ("Duration", lambda x: x.quantile(.75) - x.quantile(.25)),
    V_real_mean=("V_avg", "mean"),
    V_real_std=("V_avg", "std"),
    V_q10_mean=("V_q10", "mean"),
    Straightness_mean=("Straightness", "mean"),
    Straightness_std=("Straightness", "std"),
    Straightness_median=("Straightness", "median"),
    A_min_mean=("A_min_q20", "mean"),
    A_plus_mean=("A_plus_q80", "mean"),
    Jerk_min_mean=("Jerk_min_q20", "mean"),
    Jerk_plus_mean=("Jerk_plus_q80", "mean")
)
movement_test_level_features

Path_median  ...   Jerk_plus_mean
User ID Is Genuine Test ID               ...                 
1       False      1         288.162445  ...   4463252.129148
                   2         156.540442  ...   3036348.984542
                   3         507.016921  ...   5567486.022351
                   4         147.184662  ...   3500127.055796
                   5         181.539123  ...  65668413.160542
...                                 ...  ...              ...
88      True       6         362.420730  ...   2768040.291812
                   7         283.604623  ...   3776306.502094
                   8         297.637353  ...   2061468.363061
                   9         289.866843  ...   1199923.678115
                   10        284.718996  ...   3654754.034536

[1760 rows x 14 columns]

## 2. Mouse click and scroll features

Non-movement events are named `<button> <action>`, e.g. `left press`. I split them into `Button` and `Button Event` and count each combination.

In [5]:
df = mouse_df.copy()
click_features = df.loc[df["Event"] != "movement"]
df[["Button", "Button Event"]] = click_features["Event"].str.split(" ", expand=True)
df.groupby(by=["Button", "Button Event"]).count()

Event  Coordinates  Epoch
Button      Button Event                           
left        press         11563        11563  11563
            release       11475        11475  11475
mouse4      press             3            3      3
            release           3            3      3
right       press            71           71     71
            release          71           71     71
scrolldown  press            34           34     34
            release          34           34     34
scrollleft  press             1            1      1
            release           1            1      1
scrollright press            12           12     12
            release          12           12     12
scrollup    press           140          140    140
            release         140          140    140

Left clicks dominate (11,563 presses). Right clicks (71), the side button `mouse4` (3) and scrolling (170 scroll presses) are rare. Presses and releases don't always pair up: there are 88 more `left press` than `left release` events, so pairs have to be matched explicitly rather than assumed.

### Click dwell time

For each button, I take the next event of the same button within the attempt. If a `press` is followed by a `release`, the time between them is the click dwell time, i.e. how long the button was held. Presses without a matching release get `NaN`.

In [6]:
click_features = df.loc[df["Event"] != "movement"]
click_features.groupby([
    pd.Grouper(level="User ID"),
    pd.Grouper(level="Is Genuine"),
    pd.Grouper(level="Test ID"),
    "Button"
    ])[["Button","Button Event"]].shift(0)

Button Button Event
User ID Is Genuine Test ID Event ID Movement ID                    
1       True       1       55       <NA>          left        press
                           56       <NA>          left      release
                           74       <NA>          left        press
                           78       <NA>          left      release
                           83       <NA>          left        press
...                                                ...          ...
88      False      10      159      <NA>          left        press
                           160      <NA>          left      release
                           191      <NA>          left        press
                           192      <NA>          left      release
                           294      <NA>          left        press

[23560 rows x 2 columns]

In [7]:
MOUSE_BUTTONS = ["left", "right", "mouse4"]

button_features = df.loc[(df["Event"] != "movement") & df["Button"].isin(MOUSE_BUTTONS)].copy()
g = button_features.groupby([
    pd.Grouper(level="User ID"),
    pd.Grouper(level="Is Genuine"),
    pd.Grouper(level="Test ID"),
    "Button"
    ])
button_features["Next Button Event"] = g["Button Event"].shift(-1)
button_features["Next Epoch"] = g["Epoch"].shift(-1)

valid_press = (button_features["Button Event"] == "press") & (button_features["Next Button Event"] == "release")
button_features["Dwell Time"] = np.where(valid_press, button_features["Next Epoch"] - button_features["Epoch"], np.nan)
button_features


Event  ... Dwell Time
User ID Is Genuine Test ID Event ID Movement ID                ...           
1       True       1       55       <NA>           left press  ...   0.074006
                           56       <NA>         left release  ...        NaN
                           74       <NA>           left press  ...   0.064049
                           78       <NA>         left release  ...        NaN
                           83       <NA>           left press  ...   0.049095
...                                                       ...  ...        ...
88      False      10      159      <NA>           left press  ...   0.225703
                           160      <NA>         left release  ...        NaN
                           191      <NA>           left press  ...   0.228105
                           192      <NA>         left release  ...        NaN
                           294      <NA>           left press  ...        NaN

[23186 rows x 8 columns]

### Scroll bursts

A single turn of the scroll wheel produces a series of `scrollup`/`scrolldown` `press` events, one per notch. To count scrolling *actions* rather than notches, I look at the time between consecutive scroll presses within an attempt (`Interscroll Time`).

In [8]:
SCROLL_TYPES = [
    "scrollup",
    "scrolldown"
]
LEVELS = [
    "User ID",
    "Is Genuine",
    "Test ID"
]


scroll = df.loc[df["Button"].isin(SCROLL_TYPES) & df["Button Event"].eq("press") & (df["Event"] != "movement")].copy()
g = scroll.groupby(level=LEVELS)

scroll["Interscroll Time"] = g["Epoch"].diff()
ax = scroll["Interscroll Time"].plot(kind="box")
ax.set_ylim(ymin=0, ymax=1);


In [9]:
import seaborn as sns
import matplotlib.pyplot as plt

bins = np.logspace(
    np.log10(0.001),
    np.log10(100),
    100,
)

sns.histplot(scroll["Interscroll Time"], bins=bins)
plt.xscale("log")

plt.axvline(0.3, linestyle="--", label="0.3 s")

plt.legend();

On the log scale, most intervals are below 0.1 s, with the highest peak at about 25 ms: these are notches within one scroll. Only a handful of gaps are longer than 0.3 s, and they separate distinct scroll actions. So I use 0.3 s as the threshold: a new burst starts at the first scroll press of an attempt and after every gap longer than 0.3 s.

In [10]:
threshold = 0.3

scroll["New Burst"] = (
    scroll["Interscroll Time"].isna()
    | scroll["Interscroll Time"].gt(threshold)
)

scroll["Burst ID"] = (
    scroll.groupby(level=LEVELS, sort=False)["New Burst"]
    .cumsum()
)

scroll

Event  ... Burst ID
User ID Is Genuine Test ID Event ID Movement ID                    ...         
2       False      9       352      <NA>         scrolldown press  ...        1
                           360      <NA>         scrolldown press  ...        1
                           371      <NA>         scrolldown press  ...        1
3       False      9       247      <NA>         scrolldown press  ...        1
                           249      <NA>         scrolldown press  ...        1
...                                                           ...  ...      ...
88      True       1       95       <NA>           scrollup press  ...        1
                           97       <NA>           scrollup press  ...        1
                           99       <NA>           scrollup press  ...        1
                           101      <NA>           scrollup press  ...        1
                           103      <NA>           scrollup press  ...        1

[174 rows x 8 columns]

I join the scroll bursts and click dwell times back to the mouse events and aggregate them per attempt:
- `Burst_count`: number of scroll bursts;
- `Dwell_Time_mean`, `Dwell_Time_std`: mean and standard deviation of click dwell time;
- `Click_count`: number of clicks with a valid dwell time.

Attempts without scrolling get 0 bursts, and missing click statistics are filled with 0. Scrolling is rare (see the counts above), so `Burst_count` is 0 for most attempts.

In [11]:
mouse_df = (df
            .join(scroll[["Interscroll Time", "Burst ID"]])
            .join(button_features["Dwell Time"])
            )

In [12]:
test_level_clicks = mouse_df.groupby(level=LEVELS).agg(
    Burst_count=("Burst ID", "max"),
    Dwell_Time_mean=("Dwell Time", "mean"),
    Dwell_Time_std=("Dwell Time", "std"),
    Click_count=("Dwell Time", "count"),
).fillna({"Burst_count": 0})
# Missing click dwell is unobserved, not a zero-duration click.
test_level_clicks


Burst_count  ...  Click_count
User ID Is Genuine Test ID               ...             
1       False      1                  0  ...            6
                   2                  0  ...            6
                   3                  0  ...            5
                   4                  0  ...            6
                   5                  0  ...            3
...                                 ...  ...          ...
88      True       6                  0  ...            3
                   7                  0  ...            2
                   8                  0  ...            2
                   9                  0  ...            2
                   10                 0  ...            2

[1760 rows x 4 columns]

## 3. Keyboard features (first pass)

After `load_data`, `key_df` has one row per raw key event: the `Key`, the `Event` (`pressed` or `released`) and its `Epoch`.

In [13]:
key_df.head()

Key  ...              Epoch
User ID Is Genuine Test ID Event ID         ...                   
1       True       1       0         shift  ...   1644173829.98174
                           1             m  ...  1644173830.121021
                           2         shift  ...   1644173830.16947
                           3             m  ...   1644173830.21818
                           4             r  ...  1644173830.293582

[5 rows x 5 columns]

In [14]:
key_df["Key"].unique()

<ArrowStringArray>
[          'shift',               'm',               'r',               's',
        'spacebar',               'w',               'e',       'backspace',
               'l',               'i',             'tab',               '4',
               '2',               '8',               '9',               '1',
               '5',               '0',               '6',               '7',
               '3', 'numpadsubstract',          'rshift',         'numpad4',
         'numpad2',         'numpad8',         'numpad9',         'numpad1',
         'numpad5',         'numpad0',         'numpad6',         'numpad7',
         'numpad3',        'capslock',               '.',               'j',
               ',',               'h',               'd',               't',
               'a',               'y',               'u',               'o',
         'numlock',               'f',           'enter',               'n',
               'g',               '-',               '`',

I group the keys that reflect typing habits: numpad keys (every key whose name starts with `num`, so `numlock` is included), arrow keys, correction keys (`delete`, `backspace`) and the two shift keys.

In [15]:
NUMPAD_KEYS = [key for key in key_df["Key"].unique() if key.startswith("num")]

ARROW_KEYS = ["up", "down", "right", "left"]
CORRECTION_KEYS = ["delete", "backspace"]
SHIFT_KEYS = ["shift", "rshift"]

### Dwell time (naive)

For each key except the shift keys, I take the next event of the same key within the attempt. If a `pressed` event is followed by a `released` one, the time between them is the dwell time. Shift keys are excluded because they are often held down for a long time (see the EDA).

With auto-repeat, only the last repeated press is paired with the release, so the dwell time of a held key is underestimated. Section 5 fixes this.

In [16]:
shiftless_key_df = key_df.loc[~key_df["Key"].isin(SHIFT_KEYS)]
g = shiftless_key_df.groupby([
    pd.Grouper(level="User ID"),
    pd.Grouper(level="Is Genuine"),
    pd.Grouper(level="Test ID"),
    "Key"
    ])
key_df["Next Event"] = g["Event"].shift(-1)
key_df["Next Epoch"] = g["Epoch"].shift(-1)

valid_press = (key_df["Event"] == "pressed") & (key_df["Next Event"] == "released")
key_df["Dwell Time"] = np.where(valid_press, key_df["Next Epoch"] - key_df["Epoch"], np.nan)
key_df

Key  ... Dwell Time
User ID Is Genuine Test ID Event ID         ...           
1       True       1       0         shift  ...        NaN
                           1             m  ...   0.097159
                           2         shift  ...        NaN
                           3             m  ...        NaN
                           4             r  ...   0.061959
...                                    ...  ...        ...
88      False      10      78            2  ...   0.097268
                           79            2  ...        NaN
                           80            3  ...   0.111409
                           81            3  ...        NaN
                           82          tab  ...        NaN

[172563 rows x 8 columns]

### Flight time (naive)

Next, the time from each key press (shift excluded) to the next key press in the attempt. Strictly speaking, this is the press-to-press latency (`DD` in section 6), not the flight time used in the paper (from a release to the next press). It also includes the pauses between form fields.

In [17]:
shiftless_key_df = key_df.loc[(~key_df["Key"].isin(SHIFT_KEYS)) & (key_df["Event"] == "pressed")]
g = shiftless_key_df.groupby([
    pd.Grouper(level="User ID"),
    pd.Grouper(level="Is Genuine"),
    pd.Grouper(level="Test ID"),
    ])
key_df["Next Pressed Key"] = g["Key"].shift(-1)
key_df["Next Pressed Epoch"] = g["Epoch"].shift(-1)

key_df["Flight Time"] = key_df["Next Pressed Epoch"] - key_df["Epoch"]
key_df

Key  ... Flight Time
User ID Is Genuine Test ID Event ID         ...            
1       True       1       0         shift  ...        <NA>
                           1             m  ...    0.172561
                           2         shift  ...        <NA>
                           3             m  ...        <NA>
                           4             r  ...    0.187345
...                                    ...  ...         ...
88      False      10      78            2  ...    0.253945
                           79            2  ...        <NA>
                           80            3  ...        <NA>
                           81            3  ...        <NA>
                           82          tab  ...        <NA>

[172563 rows x 11 columns]

In [18]:
ax = key_df["Dwell Time"].plot(kind="box")
plt.axhline(0.2, linestyle="--", label="0.2 s");


In [19]:
threshold = 0.3

key_df[key_df["Dwell Time"] > threshold]["Key"].value_counts()

Key
lctrl        24
capslock      7
6             7
7             7
4             7
1             6
rctrl         6
enter         6
8             5
-             5
backspace     4
2             4
l             3
spacebar      3
numpad2       2
i             2
right         2
a             2
left          2
9             2
k             1
numpad9       1
numpad3       1
m             1
s             1
3             1
numpad6       1
h             1
g             1
0             1
n             1
'             1
super         1
alt           1
z             1
Name: count, dtype: int64

Most dwell times are around 0.1 s (the middle half lies between about 0.085 and 0.125 s), with a long tail of outliers. The keys most often held longer than 0.3 s are `lctrl`, followed by `capslock`, some digits, `rctrl` and `enter`.

In [20]:
key_df

Key  ... Flight Time
User ID Is Genuine Test ID Event ID         ...            
1       True       1       0         shift  ...        <NA>
                           1             m  ...    0.172561
                           2         shift  ...        <NA>
                           3             m  ...        <NA>
                           4             r  ...    0.187345
...                                    ...  ...         ...
88      False      10      78            2  ...    0.253945
                           79            2  ...        <NA>
                           80            3  ...        <NA>
                           81            3  ...        <NA>
                           82          tab  ...        <NA>

[172563 rows x 11 columns]

### Habit counts

Per attempt, I count the presses of keys that reflect habits rather than typing speed:
- `capslock`, `enter` and `spacebar`;
- arrow keys (summed);
- correction keys, `backspace` and `delete`;
- numpad keys (summed);
- `tab`, counted on `released` events, because its presses are almost never logged (see the EDA).

These counts use the raw events, so a key held long enough to auto-repeat is counted several times.

In [21]:
pressed_key_df = key_df[(key_df["Event"] == "pressed") & key_df["Key"].isin(["capslock", "enter", "spacebar"])]
other_keys_test_level = pressed_key_df.groupby([
    pd.Grouper(level="User ID"),
    pd.Grouper(level="Is Genuine"),
    pd.Grouper(level="Test ID"),
    "Key"
    ])["Event"].count().unstack()

In [22]:
arrow_key_df = key_df[(key_df["Event"] == "pressed") & key_df["Key"].isin(ARROW_KEYS)]
arrow_keys_test_level = arrow_key_df.groupby([
    pd.Grouper(level="User ID"),
    pd.Grouper(level="Is Genuine"),
    pd.Grouper(level="Test ID"),
    "Key"
    ])["Event"].count().unstack()
arrow_keys_test_level = arrow_keys_test_level.sum(axis=1).rename("arrows").astype("int")

In [23]:
correction_key_df = key_df[(key_df["Event"] == "pressed") & key_df["Key"].isin(CORRECTION_KEYS)]
correction_keys_test_level = correction_key_df.groupby([
    pd.Grouper(level="User ID"),
    pd.Grouper(level="Is Genuine"),
    pd.Grouper(level="Test ID"),
    "Key"
    ])["Event"].count().unstack()

In [24]:
numpad_key_df = key_df[(key_df["Event"] == "pressed") & key_df["Key"].isin(NUMPAD_KEYS)]
numpad_keys_test_level = numpad_key_df.groupby([
    pd.Grouper(level="User ID"),
    pd.Grouper(level="Is Genuine"),
    pd.Grouper(level="Test ID"),
    "Key"
    ])["Event"].count().unstack()
numpad_keys_test_level = numpad_keys_test_level.sum(axis=1).rename("numpad").astype("int")

In [25]:
released_key_df = key_df[(key_df["Event"] == "released") & key_df["Key"].isin(["tab"])]
tab_test_level = released_key_df.groupby([
    pd.Grouper(level="User ID"),
    pd.Grouper(level="Is Genuine"),
    pd.Grouper(level="Test ID"),
    "Key"
    ])["Event"].count().unstack()

Only 901 of the 1,760 attempts contain a `tab` release, so the use of `tab` for moving between fields varies a lot between attempts. That makes it a promising habit feature, as suggested in the EDA.

### Attempt-level keyboard timing

Mean and standard deviation of dwell and flight time per attempt.

In [26]:
key_test_level = key_df.groupby(level=LEVELS).agg(
    Dwell_mean=("Dwell Time", "mean"),
    Dwell_std=("Dwell Time", "std"),
    Flight_mean=("Flight Time", "mean"),
    Flight_std=("Flight Time", "std")
)

### Feature table

I join all feature families into one table with one row per attempt (88 users × 20 attempts = 1,760 rows) and 30 features. A missing count means the key wasn't pressed in that attempt, so counts are filled with 0.

In [27]:
test_level_features = (
    movement_test_level_features
    .join(test_level_clicks)
    .join(key_test_level)
    .join(other_keys_test_level)
    .join(tab_test_level)
    .join(correction_keys_test_level)
    .join(numpad_keys_test_level)
    .join(arrow_keys_test_level)
)

In [28]:
test_level_features.fillna({
    "enter": 0,
    "spacebar": 0,
    "tab": 0,
    "capslock": 0,
    "arrows": 0,
    "numpad": 0,
    "delete": 0,
    "backspace": 0
}, inplace=True)

Path_median    Path_iqr  ...  numpad  arrows
User ID Is Genuine Test ID                           ...                
1       False      1         288.162445  439.766890  ...    23.0     0.0
                   2         156.540442  126.143963  ...     0.0     0.0
                   3         507.016921  390.456178  ...    25.0     0.0
                   4         147.184662  133.094730  ...     0.0     0.0
                   5         181.539123  171.662940  ...     0.0     0.0
...                                 ...         ...  ...     ...     ...
88      True       6         362.420730  232.214024  ...     0.0     3.0
                   7         283.604623  171.941891  ...     0.0     0.0
                   8         297.637353  109.025102  ...     0.0     0.0
                   9         289.866843  104.681253  ...     0.0     0.0
                   10        284.718996   86.171983  ...     0.0     0.0

[1760 rows x 30 columns]

## 4. Baseline models

The task is verification: for a given card, decide whether an attempt was typed by the card's owner (genuine) or by someone else (impostor). As in the paper, I train one model per user on the 20 attempts on that user's card: `X[u]` holds the features and `y[u]` the `Is Genuine` labels (10 genuine, 10 impostor).

In [29]:
X = [test_level_features.loc[(user_id)] for user_id in range(1, 89)]
y = [test_level_features.loc[(user_id)].index.get_level_values("Is Genuine") for user_id in range(1, 89)]

First, a quick comparison of models on a single stratified train/test split per user, scored by accuracy and averaged over the 88 users. For reference, the paper reports a mean accuracy of 75.85% for a default decision tree on three features (average dwell time, flight time and mouse trajectory length) with a stratified 80/20 split.

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer

model = DecisionTreeClassifier(random_state=42)

scores = []
for user_id in range(1,89):
    X_train, X_test, y_train, y_test = train_test_split(X[user_id-1], y[user_id-1], stratify=y[user_id-1], test_size=.2, random_state=42)

    model.fit(X_train, y_train)
    scores.append(model.score(X_test, y_test))



single_split_results = {}

single_split_results["decision_tree"] = {"accuracy_mean": np.mean(scores), "accuracy_std": np.std(scores)}
pd.DataFrame(single_split_results).T


,accuracy_mean,accuracy_std
decision_tree,0.821023,0.172502


In [31]:
model = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    StandardScaler(),
    LogisticRegression()
)

scores = []
for user_id in range(1,89):
    X_train, X_test, y_train, y_test = train_test_split(X[user_id-1], y[user_id-1], stratify=y[user_id-1], test_size=.2, random_state=42)

    model.fit(X_train, y_train)
    scores.append(model.score(X_test, y_test))


single_split_results["logreg"] = {"accuracy_mean": np.mean(scores), "accuracy_std": np.std(scores)}
pd.DataFrame(single_split_results).T


,accuracy_mean,accuracy_std
decision_tree,0.821023,0.172502
logreg,0.892045,0.149789


In [32]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('simpleimputer', ...), ('standardscaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[bool](2,)","[False, True]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['Path_median','Path_iqr','Duration_median',...,'delete','numpad','arrows']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",True
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. Al

In [33]:
import xgboost as xgb

model = xgb.XGBClassifier(max_depth=2, random_state=42, n_jobs=1)

scores = []
for user_id in range(1,89):
    X_train, X_test, y_train, y_test = train_test_split(X[user_id-1], y[user_id-1], stratify=y[user_id-1], test_size=.2, random_state=42)

    model.fit(X_train, y_train)
    scores.append(model.score(X_test, y_test))



single_split_results["xgboost"] = {"accuracy_mean": np.mean(scores), "accuracy_std": np.std(scores)}
pd.DataFrame(single_split_results).T


,accuracy_mean,accuracy_std
decision_tree,0.821023,0.172502
logreg,0.892045,0.149789
xgboost,0.860795,0.187908


In [34]:
model.feature_importances_

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [35]:
model.feature_names_in_

array(['Path_median', 'Path_iqr', 'Duration_median', 'Duration_iqr',
       'V_real_mean', 'V_real_std', 'V_q10_mean', 'Straightness_mean',
       'Straightness_std', 'Straightness_median', 'A_min_mean',
       'A_plus_mean', 'Jerk_min_mean', 'Jerk_plus_mean', 'Burst_count',
       'Dwell_Time_mean', 'Dwell_Time_std', 'Click_count', 'Dwell_mean',
       'Dwell_std', 'Flight_mean', 'Flight_std', 'capslock', 'enter',
       'spacebar', 'tab', 'backspace', 'delete', 'numpad', 'arrows'],
      dtype='<U19')

The displayed importance vector belongs only to the last fitted XGBoost model
(user 88). It is not population-wide feature importance; there are only 16
training attempts for that model.


In [36]:
from sklearn.svm import SVC

model = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    StandardScaler(),
    SVC(kernel="rbf", gamma="auto", C=1),
)

scores = []
for user_id in range(1,89):
    X_train, X_test, y_train, y_test = train_test_split(X[user_id-1], y[user_id-1], stratify=y[user_id-1], test_size=.2, random_state=42)

    model.fit(X_train, y_train)
    scores.append(model.score(X_test, y_test))


single_split_results["svc"] = {"accuracy_mean": np.mean(scores), "accuracy_std": np.std(scores)}
pd.DataFrame(single_split_results).T


,accuracy_mean,accuracy_std
decision_tree,0.821023,0.172502
logreg,0.892045,0.149789
xgboost,0.860795,0.187908
svc,0.866477,0.180549


The current single-split results are displayed above. All models use the same
stratified 80/20 split and seed. Each user has **four** test attempts, so one
mistake changes that user's accuracy by 25 percentage points. These are
exploratory checks, not a matched reproduction of the paper.

### Cross-validation

Each attempt is tested once in stratified 5-fold CV.


In [37]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    StandardScaler(),
    LogisticRegression(C=1),
)

scores = []
for user_id in range(1,89):
    score = cross_val_score(
        model,
        X[user_id-1],
        y[user_id-1],
        cv=cv,
        n_jobs=1
    )
    scores.append(score.mean())


cv_results = {}

cv_results["logreg"] = {"accuracy_mean": np.mean(scores), "accuracy_std": np.std(scores)}
pd.DataFrame(cv_results).T


,accuracy_mean,accuracy_std
logreg,0.840341,0.093498


In [38]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    StandardScaler(),
    SVC(kernel="rbf", gamma="auto", C=1),
)

scores = []
for user_id in range(1,89):
    score = cross_val_score(
        model,
        X[user_id-1],
        y[user_id-1],
        cv=cv,
        n_jobs=1
    )
    scores.append(score.mean())


cv_results["svc"] = {"accuracy_mean": np.mean(scores), "accuracy_std": np.std(scores)}
pd.DataFrame(cv_results).T


,accuracy_mean,accuracy_std
logreg,0.840341,0.093498
svc,0.824432,0.084610


The rerun table above reports the current exploratory accuracy estimates. This
uses more features and a different split from the paper, so it does not support
a claim of a matched baseline improvement. Biometric comparisons follow in
`models.ipynb`.


## 5. Cleaning the keyboard events

For further improvement, the keyboard events have to be cleaned and transformed properly. The naive timings above pair raw events, so they are distorted by the logging quirks found in the EDA: a key held down is logged as a series of `pressed` events, and `tab` is logged (almost) only on release.

So I transform the raw events into **one row per physical key press**. `transform_key_events` is applied to the events of each attempt (`groupby(...).apply`) and goes through them in time order:
- the first `pressed` event of a key opens a press and sets its `Down` time;
- further `pressed` events of a key that is already down are treated as repeats and only collected in `Epochs`;
- the `released` event closes the press and sets its `Up` time;
- a `released` event without a preceding press (typically `tab`) becomes a press with `Down` set to the release time, no `Up` and `Release Only = True`;
- presses that are never released are kept without an `Up` time.

In [39]:
# Preserve the raw input so this cell can be rerun independently.
key_df = key.transform_key_events(raw_key_df)
key_df


Key  ...  Release Only
User ID Is Genuine Test ID               ...              
1       False      1       0      shift  ...           NaN
                           1          m  ...           NaN
                           2          r  ...           NaN
                           3          s  ...           NaN
                           4   spacebar  ...           NaN
...                                 ...  ...           ...
88      True       10      41         1  ...           NaN
                           42         0  ...           NaN
                           43       tab  ...          True
                           44         2  ...           NaN
                           45         3  ...           NaN

[84955 rows x 5 columns]

### Are the repeated presses auto-repeat?

If the repeated `pressed` events come from the operating system's key auto-repeat, the first repeat should follow the initial press after a fixed *repeat delay*, and further repeats should come at a fixed *repeat rate*. First, the time until the second press, for every press with repeats:

In [40]:
prolonged_key_df = key_df[key_df["Epochs"].transform(lambda x: len(x)>1)]
time_until_second_press = prolonged_key_df["Epochs"].transform(lambda x: x[1] - x[0]).rename("Time Until Second Press")

prolonged_key_df = prolonged_key_df.join(time_until_second_press)
prolonged_key_df["Time Until Second Press"].plot(kind="hist");

Initial repeat delays cluster around 0.25 s and 0.5 s, consistent with
auto-repeat. The next cells isolate unusually short initial intervals below
0.25 s for inspection; this threshold is descriptive and is not used by the
state machine to decide whether to merge repeated press records. The intervals
alone do not establish which hardware or operating-system settings were used.


In [41]:
quicker_df = prolonged_key_df[prolonged_key_df["Time Until Second Press"] < 0.25]
quicker_df


,,,,Key,Down,Epochs,Up,Release Only,Time Until Second Press
User ID,Is Genuine,Test ID,,,,,,,
12,False,10,58,s,1.649764e+09,"[1649764369.3014634, 1649764369.3738153]",1.649764e+09,NaN,0.072352
88,True,1,0,shift,1.649930e+09,"[1649929777.2399294, 1649929777.476989]",1.649930e+09,NaN,0.237060


In [42]:
longer_df = prolonged_key_df[prolonged_key_df["Time Until Second Press"] > 0.25]
longer_df


Key  ...  Time Until Second Press
User ID Is Genuine Test ID                ...                         
1       False      2       5       shift  ...                 0.516620
2       False      6       15      shift  ...                 0.522196
                   7       0       shift  ...                 0.511041
        True       1       15  backspace  ...                 0.496431
                   5       40  backspace  ...                 0.518165
...                                  ...  ...                      ...
88      True       5       11      shift  ...                 0.267070
                   6       11      shift  ...                 0.255065
                   7       11      shift  ...                 0.258066
                   8       11      shift  ...                 0.260082
                   10      11      shift  ...                 0.260065

[443 rows x 6 columns]

For each press with repeats, I split the intervals between consecutive `pressed` events into the initial repeat delay (the first interval) and the repeat intervals (all later ones).


In [43]:
prolonged_key_df["Press Intervals"] = (
    prolonged_key_df["Epochs"]
    .apply(lambda x: np.diff(x))
)

prolonged_key_df["Initial Repeat Delay"] = (
    prolonged_key_df["Epochs"]
    .apply(lambda x: x[1] - x[0])
)

prolonged_key_df["Repeat Intervals"] = (
    prolonged_key_df["Epochs"]
    .apply(lambda x: np.diff(x)[1:])
)
prolonged_key_df

Key  ...                                   Repeat Intervals
User ID Is Genuine Test ID                ...                                                   
1       False      2       5       shift  ...  [0.02489781379699707, 0.03800487518310547, 0.0...
2       False      6       15      shift  ...                              [0.02444934844970703]
                   7       0       shift  ...  [0.03675031661987305, 0.037627458572387695, 0....
        True       1       15  backspace  ...                              [0.04800558090209961]
                   5       40  backspace  ...  [0.022533416748046875, 0.03644442558288574, 0....
...                                  ...  ...                                                ...
88      True       5       11      shift  ...  [0.013004541397094727, 0.026005029678344727, 0...
                   6       11      shift  ...        [0.026006460189819336, 0.01200556755065918]
                   7       11      shift  ...       [0.025006532669067383, 0.014005899429321289]
                   8       11      shift  ...  [0.010988712310791016, 0.02600550651550293, 0....
                   10      11      shift  ...  [0.012004375457763672, 0.02800726890563965, 0....

[445 rows x 9 columns]

In [44]:
repeat_intervals = prolonged_key_df["Repeat Intervals"].values
squeezed_intervals = np.concatenate(repeat_intervals)
squeezed_intervals.min(), squeezed_intervals.mean(), squeezed_intervals.max()

(np.float64(0.0030024051666259766),
 np.float64(0.02774692743445764),
 np.float64(0.051995277404785156))

In [45]:
sns.histplot(squeezed_intervals);

The repeat intervals are short and regular: 3–52 ms, about 28 ms on average (roughly 36 repeats per second), which is typical for keyboard auto-repeat. The histogram has peaks at about 12, 25 and 37 ms, i.e. multiples of roughly 12 ms, the same as the base interval between mouse events. The logger seems to record events on a ~12 ms tick; if so, timings are only precise to about 12 ms, despite the microsecond timestamps.

**Conclusion:** the repeated `pressed` events are auto-repeat, so merging them into one physical press is correct. A key's dwell time runs from its first `pressed` event to its `released` event.

### Presses without a release

Which keys have a press without a release (no `Up` time)?

In [46]:
orphans_df = key_df[key_df["Up"].isna()]
orphans_df["Key"].unique()

<ArrowStringArray>
['tab']
Length: 1, dtype: str

In [47]:
tab_only_df = key_df[key_df["Key"] == "tab"]
tab_only_df.isna().sum()

Key                0
Down               0
Epochs             0
Up              3169
Release Only      19
dtype: int64

Only `tab`. It has 3,188 presses, of which 3,169 are release-only (no `pressed` event was logged) and only 19 have both events. This confirms the EDA finding that the `tab` key-down is almost never logged. Release-only rows have no real `Down` time (it is set to the release time), so their dwell time is undefined (`NaN`).

## 6. Keyboard features on the cleaned events

### Timing features

For each press I take the next press in the same attempt and compute the standard keystroke timings:
- `Dwell Time` = `Up − Down`: how long the key was held;
- `Flight Time` = next `Down` − `Up`: from the release to the next press, as in the paper. It is negative when the next key goes down before this one is released (rollover);
- `DD` = next `Down` − `Down`: press-to-press latency;
- `UU` = next `Up` − `Up`: release-to-release latency.
Release-only rows are excluded from both sides of next-press timings. Their
synthetic ordering timestamp does not count as an observed key-down.


In [48]:
# Release-only rows remain available for counts but are excluded from timings.
featured_key_df = key.add_key_dynamics(raw_key_df)
featured_key_df.head()


Key          Down  ...        DD  Flight Time
User ID Is Genuine Test ID                            ...                       
1       False      1       0     shift  1.644176e+09  ...  0.089481    -0.112323
                           1         m  1.644176e+09  ...  0.215051     0.152125
                           2         r  1.644176e+09  ...  0.151712     0.088371
                           3         s  1.644176e+09  ...  0.127017     0.026023
                           4  spacebar  1.644176e+09  ...  0.176681     0.101435

[5 rows x 9 columns]

In [49]:
featured_key_df[["Key", "Dwell Time", "Flight Time", "DD", "UU"]].head()


Key  Dwell Time  ...        DD        UU
User ID Is Genuine Test ID                          ...                    
1       False      1       0     shift    0.201804  ...  0.089481 -0.049397
                           1         m    0.062926  ...  0.215051  0.215467
                           2         r    0.063342  ...  0.151712  0.189364
                           3         s    0.100993  ...  0.127017  0.101269
                           4  spacebar    0.075246  ...  0.176681  0.315332

[5 rows x 5 columns]

### Habit counts and attempt-level timing

The `capslock`, `enter` and `spacebar` counts are recomputed on the cleaned presses, so each physical press counts once, however long the key was held.

In [50]:
other_key_df = featured_key_df[featured_key_df["Key"].isin(["capslock", "enter", "spacebar"])]
other_keys_test_level = other_key_df.groupby(LEVELS + ["Key"]).size().unstack(fill_value=0)
other_keys_test_level


Key                         capslock  enter  spacebar
User ID Is Genuine Test ID                           
1       False      1               0      0         2
                   2               0      0         2
                   3               8      0         2
                   4               0      0         2
                   5               0      0         2
...                              ...    ...       ...
88      True       6               0      0         2
                   7               0      0         3
                   8               0      0         2
                   9               2      0         2
                   10              0      0         2

[1759 rows x 3 columns]

Per attempt, I summarize dwell and flight times with the median and the IQR instead of the mean and standard deviation. Flight time includes the long pauses between form fields, and a few such pauses would dominate the mean.

In [51]:
key_test_level = featured_key_df.groupby(level=LEVELS).agg(
    Dwell_median=("Dwell Time", "median"),
    Dwell_iqr=("Dwell Time", lambda x: x.quantile(.75) - x.quantile(.25)),
    Flight_median=("Flight Time", "median"),
    Flight_iqr=("Flight Time", lambda x: x.quantile(.75) - x.quantile(.25)),
    DD_median=("DD", "median"),
    DD_iqr=("DD", lambda x: x.quantile(.75) - x.quantile(.25)),
    UU_median=("UU", "median"),
    UU_iqr=("UU", lambda x: x.quantile(.75) - x.quantile(.25))
)
key_test_level

Dwell_median  Dwell_iqr  ...  UU_median    UU_iqr
User ID Is Genuine Test ID                           ...                     
1       False      1            0.075183   0.024579  ...   0.216582  0.320033
                   2            0.086801   0.014752  ...   0.468952  0.786050
                   3            0.073625   0.025564  ...   0.188255  0.445632
                   4            0.121413   0.021379  ...   0.206442  1.025717
                   5            0.170910   0.049758  ...   0.218171  0.293737
...                                  ...        ...  ...        ...       ...
88      True       6            0.142035   0.059016  ...   0.404103  1.009757
                   7            0.141037   0.053761  ...   0.491128  1.105286
                   8            0.141034   0.053261  ...   0.578167  0.787192
                   9            0.141034   0.059767  ...   0.700176  0.960236
                   10           0.147538   0.066015  ...   0.631155  0.758201

[1760 rows x 8 columns]

In [52]:
ax = featured_key_df["Dwell Time"].plot(kind="box")
plt.axhline(0.2, linestyle="--", label="0.2 s");


After cleaning, most dwell times lie between about 0.08 and 0.13 s. The long tail remains (up to more than 3 s), now including the full hold time of auto-repeated keys.

### Shift and correction keys

How long a user holds `shift` may be a habit of its own, so I compute the shift dwell statistics and count per attempt, and the same for the correction keys (`backspace`, `delete`).

In [53]:
shift_key_df = featured_key_df[featured_key_df["Key"].isin(SHIFT_KEYS)]
shift_key_features = shift_key_df.groupby(level=LEVELS).agg(
    Shift_Dwell_mean=("Dwell Time", "mean"),
    Shift_Dwell_std=("Dwell Time", "std"),
    Shift_count=("Dwell Time", "count")
)

In [54]:
correction_key_df = featured_key_df[featured_key_df["Key"].isin(CORRECTION_KEYS)]
correction_key_features = correction_key_df.groupby(level=LEVELS).agg(
    Corr_Dwell_mean=("Dwell Time", "mean"),
    Corr_Dwell_std=("Dwell Time", "std"),
    Corr_count=("Dwell Time", "count")
)

### Updated feature table

Use the shared package builder for the corrected comparison: 18 keyboard
features plus 21 mouse features. Habit counts use physical actions, and shift
statistics have stable names. Earlier tables remain part of the exploration.


In [55]:
corrected_feature_sets = build_feature_sets(raw_mouse_df, raw_key_df)
test_level_features = corrected_feature_sets["keys_mouse"]
assert len(test_level_features) == 1760
test_level_features


Dwell_median  ...  Click_count
User ID Is Genuine Test ID                ...             
1       False      1            0.075183  ...            6
                   2            0.086801  ...            6
                   3            0.073625  ...            5
                   4            0.121413  ...            6
                   5            0.170910  ...            3
...                                  ...  ...          ...
88      True       6            0.142035  ...            3
                   7            0.141037  ...            2
                   8            0.141034  ...            2
                   9            0.141034  ...            2
                   10           0.147538  ...            2

[1760 rows x 39 columns]

In [56]:
X = [test_level_features.loc[(user_id)] for user_id in range(1, 89)]
y = [test_level_features.loc[(user_id)].index.get_level_values("Is Genuine") for user_id in range(1, 89)]

### Re-evaluation

Recompute exploratory accuracy using stratified 5-fold CV on the corrected
combined table. Fixed historical settings are retained; they were not selected
using an independent development dataset, so these are not final tuned scores.


In [57]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    StandardScaler(),
    LogisticRegression(),
)

scores = []
for user_id in range(1,89):
    score = cross_val_score(
        model,
        X[user_id-1],
        y[user_id-1],
        cv=cv,
        n_jobs=1
    )
    scores.append(score.mean())

np.mean(scores), np.std(scores)

(np.float64(0.8522727272727273), np.float64(0.09940421694831379))

The same folds are used for an RBF SVC with the historical `gamma="auto", C=3`
settings. Compare the freshly computed outputs rather than saved historical
numbers. The model notebook uses fixed baseline settings and biometric metrics.


In [58]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    StandardScaler(),
    SVC(kernel="rbf", gamma="auto", C=3),
)

scores = []
for user_id in range(1,89):
    score = cross_val_score(
        model,
        X[user_id-1],
        y[user_id-1],
        cv=cv,
        n_jobs=1
    )
    scores.append(score.mean())

np.mean(scores), np.std(scores)

(np.float64(0.8539772727272728), np.float64(0.07277830240980405))

## Summary

- Physical presses merge auto-repeat events; release-only navigation events are
  retained for counts and excluded from timing calculations.
- Mouse geometry retains tied-time coordinates and leaves undefined rates
  missing; unobserved click dwell is not replaced by zero.
- The corrected table is built by the package. Its outputs above replace the
  older accuracy claims; no matched improvement over the paper is asserted.
- `models.ipynb` compares indexed scores under explicit P1/P2 protocols and
  reports pooled and per-user metrics separately.
